# ML-03 â€” Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/broskell/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 2: Refresh / Content Opportunity Scoring — framed as binary classification for ranking.**

The task is to classify each content page as *declining* or *not declining*, using observable
90-day search and engagement signals. The model outputs a probability (0 to 1) that a page is
at risk. Those probabilities are sorted to produce a **ranked review queue** — the top-K pages
are the ones an editor should check first.

Classification is the right fit because the decision is binary (review this page or skip it),
and the ranked output respects limited editorial capacity. The starter dataset ships a proxy
label (`trend_direction == "down"`) that gives us a target to train against immediately.

The starter pipeline already proved this works: the random forest scored Precision@50 of 0.68
against a rule-baseline of 0.24 (from the committed `outputs/model_report.md`). That means
handing an editor 34 correct pages out of 50 instead of 12.

In [3]:
# No code needed for framing — this cell confirms the task type.
# The mapping is: binary classification problem -> probability ranking -> top-K action queue.
print("ML task type: binary classification, used for ranking.")
print("Input: 90-day observable signals per page.")
print("Output: probability(declining) -> sorted queue -> reviewer acts on top-50.")


ML task type: binary classification, used for ranking.
Input: 90-day observable signals per page.
Output: probability(declining) -> sorted queue -> reviewer acts on top-50.


## 2. Target or proxy

**Target: `is_declining_label` — a proxy, not an observed outcome.**

The starter dataset defines the label as:

```
is_declining_label = (trend_direction == "down")
```

`trend_direction` is computed from `trend_pct`, which compares impressions in the most recent
30 days to the 30 days before that. A page is "down" if impressions dropped by more than 20%.

This is a **proxy label** — it is derived from the SAME 90-day window as the features, not from
a future time window. That means:

- The model learns patterns associated with decline in the current window — useful for
  prioritizing review, but not for predicting what happens next.
- A stronger capstone would split the timeline: features from days 1-60 predicting decline
  in days 61-90, or features from the prior 90 days predicting decline in the next 30.

I am honest about this: the model ranks pages by current decline signals. That is still
valuable for an editor deciding where to look first.

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Build the label exactly as the starter pipeline does
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('=== Target: is_declining_label ===')
print(f'Positive (declining):  {(df["is_declining_label"]==1).sum():>6,}  ({(df["is_declining_label"]==1).mean()*100:.1f}%)')
print(f'Negative (not):        {(df["is_declining_label"]==0).sum():>6,}  ({(df["is_declining_label"]==0).mean()*100:.1f}%)')
print()
print('Value counts:')
print(df['is_declining_label'].value_counts())
print()
print('=== Breakdown by trend_direction (label source) ===')
for val, cnt in df['trend_direction'].value_counts().items():
    label = 1 if val == 'down' else 0
    print(f'  {val:8s} -> label={label}  {cnt:>6,}  ({cnt/len(df)*100:.1f}%)')
print()
print('Note: The label is a PROXY from the current 90-day window.')
print('It is NOT a future outcome. This is fine for decision-support ranking.')


=== Target: is_declining_label ===
Positive (declining):  16,262  (54.2%)
Negative (not):        13,738  (45.8%)

Value counts:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

=== Breakdown by trend_direction (label source) ===
  down     -> label=1  16,262  (54.2%)
  stable   -> label=0   5,962  (19.9%)
  up       -> label=0   4,388  (14.6%)
  new      -> label=0   2,236  (7.5%)
  flat     -> label=0   1,152  (3.8%)

Note: The label is a PROXY from the current 90-day window.
It is NOT a future outcome. This is fine for decision-support ranking.


## 3. Success metric

**Precision@50 — the fraction of my top-50 recommendations that are actually declining.**

Why Precision@K and not ROC AUC or accuracy:

- An editor can review about 50 pages per cycle. They do not care about the full ranking —
  they care about the quality of the first screen of results.
- ROC AUC measures discrimination across the whole dataset. A model can score AUC=0.80 and
  still have a terrible top-50 if it cannot separate the highest-risk pages from the noise.
- Precision@50 directly answers the question: *"If I hand a reviewer 50 pages, how many are
  actually worth their time?"*

The starter pipeline baseline (hand-tuned rules) scored Precision@50 of 0.24 — about 12 of
50. The random forest reached 0.68 — about 34 of 50. That is a meaningful improvement for
a reviewer's workflow.

I will also track average precision (AP) as a secondary metric — it summarizes precision
across all recall levels and is useful when comparing models.

In [7]:
# Demonstrate why Precision@K matters more than accuracy for this problem
pos_rate = df['is_declining_label'].mean()
print(f'Base rate (declining): {pos_rate*100:.1f}%')
print(f'A model that always predicts "declining" gets {pos_rate*100:.1f}% accuracy.')
print(f'But its Precision@50 is the base rate — same as random guessing from the pool.')
print()
print('What Precision@50 = 0.68 means (starter random forest, from model_report.md):')
print(f'  ~{0.68*50:.0f} of the top 50 pages are genuinely declining.')
print('What Precision@50 = 0.24 means (baseline rules):')
print(f'  ~{0.24*50:.0f} of the top 50 pages are genuinely declining.')
print(f'Difference: ~{int((0.68-0.24)*50)} more correct recommendations per cycle.')


Base rate (declining): 54.2%
A model that always predicts "declining" gets 54.2% accuracy.
But its Precision@50 is the base rate — same as random guessing from the pool.

What Precision@50 = 0.68 means (starter random forest, from model_report.md):
  ~34 of the top 50 pages are genuinely declining.
What Precision@50 = 0.24 means (baseline rules):
  ~12 of the top 50 pages are genuinely declining.
Difference: ~22 more correct recommendations per cycle.


## 4. The unit of analysis, as a real dataframe

**One row represents one pseudonymized content item (a page).**

The starter dataset is aggregated: each row is a single page, with all metrics summed or
averaged over a trailing 90-day window. There are no daily rows — aggregation is already
done. This is a row-level (tabular) classification problem, not a time-series one.

Below: the actual dataframe loaded from `data/raw/content_refresh_anonymized.csv`.

In [9]:
print('=== Unit of analysis ===')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print()
print('One row = one pseudonymized content item (page).')
print()
print('--- df.head() ---')
display_cols = ['content_id', 'client_id', 'content_type', 'impressions_90d',
                'clicks_90d', 'sessions_90d', 'ctr', 'avg_position',
                'content_age_days', 'days_since_last_update', 'trend_direction']
print(df[display_cols].head().to_string(index=False))
print()
print('--- All columns ---')
for c in df.columns:
    print(f'  {c}')
print()
print('--- Relevant columns for Lane 2 (Refresh Scoring) ---')
key_cols = [
    'content_id', 'client_id',           # identifiers
    'content_type', 'main_intent',       # content metadata
    'impressions_90d', 'clicks_90d',     # search performance
    'sessions_90d', 'engaged_sessions_90d',  # engagement
    'ctr', 'avg_position',              # click-through context
    'content_age_days', 'days_since_last_update',  # freshness
    'engagement_rate', 'scroll_rate',   # engagement depth
    'trend_direction', 'trend_pct',     # trend (target source — NOT features)
]
print('Key columns:', key_cols)


=== Unit of analysis ===
Shape: 30,000 rows x 45 columns

One row = one pseudonymized content item (page).

--- df.head() ---
          content_id         client_id    content_type  impressions_90d  clicks_90d  sessions_90d  ctr  avg_position  content_age_days  days_since_last_update trend_direction
content_304f48230142 client_f369cb89fc keyword article             3803          29            17 0.76          10.6               187                      20            down
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7             9 0.05          20.3               445                      25            down
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          11            11 0.09          36.5               141                      20            down
content_331d6c4de07b client_19581e27de keyword article            11751          58            78 0.49           6.2               463                      22          stable

## 5. Why ML beats a fixed rule here

A fixed-rule baseline — like the starter's hand-tuned weighted score — gets about 12 of its
top 50 recommendations correct. A random forest gets about 34 (Precision@50 = 0.68, from the
committed model_report.md). That gap is not small; it comes from a real problem that hand-tuned
rules cannot solve:

1. **Interacting signals.** A page with 500 impressions, position 8, and 180-days stale is
   riskier than a page with 5,000 impressions, position 3, and same staleness. Rules that
   chain `if/and/then` conditions miss these interactions — ML captures them.

2. **Many dimensions at once.** The starter ships 44 columns. A human can tune thresholds on
   3 or 4 at most before the rule becomes unmaintainable. ML handles all of them together.

3. **Thresholds should be learned, not guessed.** Is 500 impressions the right floor? Should
   180 days of staleness trigger review? ML learns these cutoffs from the data by optimizing
   against the outcome, not from someone's intuition.

4. **Client heterogeneity.** Different clients have different traffic profiles, content
   strategies, and history lengths. A single rule cannot adapt — ML with client-holdout
   validation learns patterns that generalize across unseen clients.

A fixed rule is still useful: it is the baseline. If ML cannot beat it, the extra complexity
is not worth it. In this case, the starter pipeline already showed it can — by a wide margin.

In [11]:
# Show the starter pipeline's own evidence that ML beats rules
print('=== Starter pipeline results (from outputs/model_report.md) ===')
print()
print('Precision@50 comparison:')
print('  Baseline (hand-tuned rules):     0.24  (~12 of 50 correct)')
print('  Logistic regression:             0.40  (~20 of 50 correct)')
print('  Decision tree:                   0.62  (~31 of 50 correct)')
print('  Random forest (best):            0.68  (~34 of 50 correct)')
print()
print('ROC AUC comparison:')
print('  Baseline:  0.627')
print('  Random forest: 0.747')
print()
print('The gap between 0.24 and 0.68-0.74 is the reason ML earns its complexity here.')
print('Note: these are verified numbers from the committed model_report.md.')


=== Starter pipeline results (from outputs/model_report.md) ===

Precision@50 comparison:
  Baseline (hand-tuned rules):     0.24  (~12 of 50 correct)
  Logistic regression:             0.40  (~20 of 50 correct)
  Decision tree:                   0.62  (~31 of 50 correct)
  Random forest (best):            0.68  (~34 of 50 correct)

ROC AUC comparison:
  Baseline:  0.627
  Random forest: 0.747

The gap between 0.24 and 0.68-0.74 is the reason ML earns its complexity here.
Note: these are verified numbers from the committed model_report.md.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.